# Head Output Decomposition

Decompose each head's contribution: residual alignment, logit projection,
value source decomposition, interference analysis, and rank.

## Why This Matters

Composition head output decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.head_output_decomposition import (
    head_residual_contribution, head_logit_projection,
    head_value_decomposition, head_output_interference,
    head_output_rank_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Residual Contribution

How much does each head contribute to the residual stream?

In [ ]:
for layer in range(4):
    result = head_residual_contribution(model, tokens, layer=layer)
    print(f"Layer {layer} (resid norm={result['residual_norm']:.4f}):")
    for h in result['per_head']:
        sign = '+' if h['is_constructive'] else '-'
        print(f"  H{h['head']}: norm={h['mean_norm']:.4f} "
              f"({h['fraction_of_residual']:.1%}), "
              f"align={h['alignment_to_residual']:+.4f} [{sign}]")
    print()

## Logit Projection

Project each head's output through the unembedding to see which tokens
each head promotes or suppresses.

In [ ]:
result = head_logit_projection(model, tokens, layer=2, position=-1)
print(f"Layer {result['layer']}, position {result['position']}:\n")
for h in result['per_head']:
    promoted = ', '.join(f"{t['token']}:{t['logit']:+.3f}" for t in h['top_promoted'][:3])
    suppressed = ', '.join(f"{t['token']}:{t['logit']:+.3f}" for t in h['top_suppressed'][:3])
    print(f"  H{h['head']} (norm={h['logit_norm']:.4f}):")
    print(f"    promotes: {promoted}")
    print(f"    suppresses: {suppressed}")

## Value Decomposition

Which source positions contribute most to a head's output?

In [ ]:
result = head_value_decomposition(model, tokens, layer=2, head=0)
for q in result['per_query']:
    sources = ', '.join(
        f"pos{s['source']}({s['attention_weight']:.3f}→{s['contribution_norm']:.4f})"
        for s in q['per_source'][:3]
    )
    print(f"  Query pos {q['query_position']}: top_src=pos{q['top_source']} [{sources}]")

## Output Interference

Do heads cooperate (constructive) or compete (destructive)?

In [ ]:
for layer in range(4):
    result = head_output_interference(model, tokens, layer=layer)
    mode = 'constructive' if result['is_mostly_constructive'] else 'destructive'
    print(f"Layer {layer}: ratio={result['interference_ratio']:.4f} [{mode}], "
          f"{result['n_constructive_pairs']}/{len(result['pairs'])} constructive pairs")

## Output Rank Analysis

Effective rank of each head's output — low rank = simple, structured output.

In [ ]:
for layer in range(4):
    result = head_output_rank_analysis(model, tokens, layer=layer)
    print(f"Layer {layer} (mean rank={result['mean_effective_rank']:.2f}):")
    for h in result['per_head']:
        rank_bar = '█' * int(min(h['effective_rank'], 5))
        lr = ' [LOW RANK]' if h['is_low_rank'] else ''
        print(f"  H{h['head']}: eff_rank={h['effective_rank']:.2f}, "
              f"top_sv={h['top_sv_fraction']:.2%} {rank_bar}{lr}")
    print()